In [ ]:
import ast
import os
import warnings

import pandas as pd

from animal_re_id.config import load_paths, resolve_path
from animal_re_id.datasets import TurtlesOfSMSRC
from animal_re_id.models import best_matches_discovery, best_matches_matching, gather_df, top_indices
from animal_re_id.similarity import SimilarityCache, get_similarity_dataset_config

warnings.filterwarnings("ignore")

In [ ]:
local_path = "../../config/paths.json"
paths = load_paths("TurtlesOfSMSRC", local_path=local_path)

B_max = 100
max_per_method = 80
max_per_identity = 2

root = resolve_path(paths["root"])
dataset_name = "TurtlesOfSMSRC"
path_replace_identities = "hidden/matched_encounters.txt"
device = "cuda"

config = get_similarity_dataset_config(dataset_name, B=B_max)
img_load = config.img_load
matching_parts = config.matching_parts
method_names = [spec.name for spec in config.pipeline_specs]

In [ ]:
root_results = resolve_path(paths["root_similarity"])
cache = SimilarityCache(root_results)

In [ ]:
with open(path_replace_identities) as f:
    replace_identity = ast.literal_eval(f.read())

os.makedirs(root_results, exist_ok=True)
dataset_kwargs = {"img_load": img_load, "replace_identity": replace_identity}
database_query_matching = False
dataset = TurtlesOfSMSRC(root, database_query_matching, dataset_kwargs=dataset_kwargs)

In [ ]:
results_all = {}
query_all = {}
database_all = {}
ignore_policy = config.ignore_pairs_policies[0]
split_name = config.splitter.get_split_names()[0]
for matching_part in matching_parts:
    results = None
    for pipeline_spec in config.pipeline_specs:
        saved_vars = cache.load_result(pipeline_spec, ignore_policy, split_name, img_load, matching_part)
        similarity = saved_vars["similarity"]
        query = saved_vars["query"]
        database = saved_vars["database"]

        ignore_lower_diagonal = False if database_query_matching else True
        i_sort, j_sort = top_indices(similarity, ignore_lower_diagonal=ignore_lower_diagonal)
        result = gather_df(query, database, i_sort, j_sort, similarity, pipeline_spec.name, matching_part)
        results = pd.concat((results, result))

    # TODO: unify it somehow so that it is general everywhere
    max_per_encounter = 4
    if database_query_matching:
        results_all[matching_part] = best_matches_matching(results, max_per_encounter)
    else:
        results_all[matching_part] = best_matches_discovery(results, max_per_method * len(method_names), max_per_identity)

In [ ]:
results_save = pd.concat(results_all.values())
if "key" in results_save.columns:
    results_save = results_save.drop(["key"], axis=1)
results_save.to_csv(f"../../verification_data_{dataset_name}.csv", index=False)

In [ ]:
from animal_re_id.models import plot_matches

dataset = TurtlesOfSMSRC(root, database_query_matching, dataset_kwargs=dataset_kwargs)
query_all = dataset.wildlife_dataset

n_plot = 5
for matching_part in matching_parts:
    for _, result in results_all[matching_part].iloc[:n_plot].iterrows():
        plot_matches(query_all, result)